# LightGBM unit commitment

**TDT4173 Modern Machine Learning in Practice, Group 61**  
**Authors:** Fredrik Nymoen (106988) and Jonas Birkeli (107006)  
**Kaggle team:** [61] Nymoen Birkeli

Run all cells in a fresh Python kernel. This notebook reads only the supplied raw
course files, trains from scratch on all 2,922 labeled cases from 2015 to 2022,
and predicts the 725 cases from 2023 to 2024. Each case contains 168 hours and
14 generators. No installation, downloads, saved models or other notebooks are used.
Dependencies are pinned in `requirements_submission.txt` (Python 3.14.7).

Data files: `data/kernel/`, `data/extended/` and `prediction_mapping.csv`, relative to the notebook folder.

This notebook fits 14 separate binary classifiers, one per generator. Each receives 361 inputs
(or 370 for generators in plants with multiple units). The output is
`outputs/lightgbm_submission.csv`.


## Fixed training settings

Tree counts are rounded medians of the 2020/2021 selection runs. The
classifiers use 31 leaves, at least 100 rows per leaf and L2 regularization
of 10. The report explains the feature and hyperparameter selection.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from time import perf_counter
import gc
import hashlib
import platform
import re
import shutil
import numpy as np
import pandas as pd
import yaml
from IPython.display import display
# Locate the folder containing the supplied raw data.
ROOT = Path.cwd()
for folder in [ROOT, *ROOT.parents]:
    if (folder / 'data/kernel/Unit_commitment_decisions.csv').is_file():
        ROOT = folder
        break
HOURS = 168
SEED = 42
THREADS = 4
# Fixed LightGBM settings selected during development.
PARAMETERS = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'min_data_in_leaf': 100,
    'lambda_l2': 10.0,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'max_bin': 127,
    'num_threads': 4,
    'deterministic': True,
    'force_col_wise': True,
    'verbosity': -1,
}
# Number of boosting rounds for each generator.
TREE_COUNTS = {
    'Hogga_G1': 214,
    'Tokke_G1': 455,
    'Tokke_G2': 250,
    'Tokke_G3': 352,
    'Tokke_G4': 384,
    'Lio_G1': 228,
    'Byrte_G1': 272,
    'Vinje_G1': 279,
    'Vinje_G2': 200,
    'Vinje_G3': 316,
    'Haukeli_G1': 109,
    'Songa_G1': 228,
    'Kjela_G1': 377,
    'Vesle Kjela_G1': 232,
}
# Output name and fingerprint of the selected submission.
OUTPUT_PATH = ROOT / 'outputs/lightgbm_submission.csv'
EXPECTED_CSV_SHA256 = 'f6bc9a0c2db660dbc6ce771ee3b5c2020f43ea31cd75eaa7ff27cf730191e830'
print('Python:', platform.python_version())
print({name: metadata.version(name) for name in ('lightgbm', 'numpy', 'pandas', 'scipy', 'PyYAML')})
display(pd.DataFrame({'generator': list(TREE_COUNTS), 'trees': list(TREE_COUNTS.values())}))


## Raw data and hydraulic topology

Cases are joined by UTC timestamps and named reservoirs, plants, rivers and
tunnels. Initial storage comes from case start. Prices, inflows and constraints
cover the week. Terminal water values come from the end of the week.
Observed storage after case start and optimizer calculation time are excluded.


In [ ]:
# Each row identifies one planning week. Preserve the mapping's row order.
RAW_FILES = {
    'labels': ROOT / 'data/kernel/Unit_commitment_decisions.csv',
    'mapping': ROOT / 'prediction_mapping.csv',
    'prices': ROOT / 'data/kernel/Historical_day_ahead_price_2015_2025.csv',
    'inflows': ROOT / 'data/kernel/Historical_inflow_1958_2025.csv',
    'volumes': ROOT / 'data/kernel/Historical_volume_2015_2024.csv',
    'water_values': ROOT / 'data/kernel/Synthetic_water_value_2015_2024.csv',
    'min_flow': ROOT / 'data/extended/Constraint_min_flow.csv',
    'min_volume': ROOT / 'data/extended/Constraint_min_volume.csv',
    'topology': ROOT / 'data/extended/Tokke_Vinje_topology.yaml',
}
train_cases = pd.read_csv(RAW_FILES['labels'])
test_cases = pd.read_csv(RAW_FILES['mapping'])
for cases in (train_cases, test_cases):
    cases['starttime'] = pd.to_datetime(cases['starttime'], utc=True)
    if cases['Run No'].duplicated().any() or cases['starttime'].isna().any():
        raise ValueError('Each case needs a unique Run No and a valid start time.')
train_cases = train_cases.sort_values('starttime').reset_index(drop=True)
# The label file already defines the required output column order.
LABELS = [name for name in train_cases if name.startswith('result_committed_')]
TARGETS = {}
for position, name in enumerate(LABELS):
    generator, hour = re.fullmatch('result_committed_(.+_G\\d+)_t(\\d+)', name).groups()
    TARGETS.setdefault(generator, {})[int(hour)] = (name, position)
GENERATORS = list(TARGETS)
TARGET_COLUMNS = {g: [TARGETS[g][h][0] for h in range(HOURS)] for g in GENERATORS}
TARGET_POSITIONS = {g: [TARGETS[g][h][1] for h in range(HOURS)] for g in GENERATORS}
if len(GENERATORS) != 14 or len(LABELS) != 2352:
    raise ValueError('Expected 14 generators and 168 targets per generator.')
if not train_cases[LABELS].isin([0, 1]).all().all():
    raise ValueError('Training targets must be binary and nonmissing.')

def labels_for(cases, generator):
    """Flatten one generator's training labels in case/hour order."""
    return cases[TARGET_COLUMNS[generator]].to_numpy(dtype=np.uint8).reshape(-1)
all_starts = pd.DatetimeIndex(pd.concat([train_cases.starttime, test_cases.starttime]))
FIRST_START = all_starts.min()
LAST_END = all_starts.max() + pd.Timedelta(hours=HOURS)
print(f'{len(train_cases)} training cases, {len(test_cases)} test cases, {len(LABELS)} targets')


In [ ]:
def read_timeseries(key):
    """Read an input table indexed by UTC timestamps."""
    frame = pd.read_csv(RAW_FILES[key])
    dates = pd.to_datetime(frame.pop('date'), utc=True, errors='raise')
    if dates.isna().any() or dates.duplicated().any():
        raise ValueError(f'Invalid or duplicate timestamps in {key}')
    frame.index = pd.DatetimeIndex(dates, name='date')
    frame = frame.sort_index().loc[FIRST_START:LAST_END]
    frame = frame.apply(pd.to_numeric, errors='raise').astype(np.float32)
    if frame.empty or not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f'Empty or nonfinite input: {key}')
    return frame
tables = {key: read_timeseries(key) for key in ('prices', 'inflows', 'volumes', 'water_values', 'min_flow', 'min_volume')}
with RAW_FILES['topology'].open(encoding='utf-8') as stream:
    topology = yaml.safe_load(stream)
RESERVOIRS = list(tables['volumes'].columns)
CAPACITY = {r: float(topology['model']['reservoir'][r]['max_vol']) for r in RESERVOIRS}
hourly_required = pd.date_range(FIRST_START, LAST_END, freq='h', inclusive='left')
for key in ('prices', 'inflows', 'min_flow', 'min_volume'):
    used = tables[key].loc[tables[key].index < LAST_END]
    if not used.index.equals(hourly_required):
        raise ValueError(f'{key} must contain exactly one record per required hour.')
for key, timestamps in (('volumes', all_starts), ('water_values', all_starts + pd.Timedelta(hours=HOURS))):
    if not timestamps.difference(tables[key].index).empty:
        raise ValueError(f'Missing required daily records in {key}')
connections = topology['connections']

def supplying_reservoirs(name, kind, visited=None):
    """Follow intake connections backwards through the supplied tunnels."""
    visited = set() if visited is None else visited
    node = (kind, name)
    if node in visited:
        raise ValueError('Cycle in intake topology.')
    if kind == 'reservoir':
        return {name}
    result = set()
    for edge in connections:
        if edge['to'] == name and edge['to_type'] == kind and (edge['from_type'] in {'reservoir', 'tunnel'}):
            result |= supplying_reservoirs(edge['from'], edge['from_type'], visited | {node})
    return result
FEEDERS = {}
for generator in GENERATORS:
    plants = [
        e['to']
        for e in connections
        if e['from'] == generator and e['from_type'] == 'generator' and (e['to_type'] == 'plant')
    ]
    FEEDERS[generator] = sorted(supplying_reservoirs(plants[0], 'plant'))
# Traverse every supplied hydraulic predecessor, including river reaches and plants.
# A visited set prevents repeated counting at merges or cycles. These are possible
# upstream connections, not evidence that all upstream water arrives in this horizon.

def upstream_reservoirs(generator):
    """Find all reservoirs upstream of one generator."""
    plant = next((
        e['to']
        for e in connections
        if e['from'] == generator and e['from_type'] == 'generator' and (e['to_type'] == 'plant')
    ))
    pending, visited, reservoirs = ([('plant', plant)], set(), set())
    while pending:
        kind, name = pending.pop()
        if (kind, name) in visited:
            continue
        visited.add((kind, name))
        if kind == 'reservoir':
            reservoirs.add(name)
        pending.extend((
            (edge['from_type'], edge['from'])
            for edge in connections
            if (
                edge['to_type'] == kind
                and edge['to'] == name
                and (edge['from_type'] in {'reservoir', 'river', 'tunnel', 'plant'})
            )
        ))
    return sorted(reservoirs)
UPSTREAM = {g: upstream_reservoirs(g) for g in GENERATORS}
OUTLET_ELEVATION = {}
for g in GENERATORS:
    plant = next((
        e['to']
        for e in connections
        if e['from'] == g and e['from_type'] == 'generator' and (e['to_type'] == 'plant')
    ))
    OUTLET_ELEVATION[g] = float(topology['model']['plant'][plant]['outlet_line'])
# Directed hydraulic edges exclude generator membership edges. Stop at the first
# receiving reservoir; never interpret graph connectivity as realized routing.
HYDRAULIC_KINDS = {'reservoir', 'plant', 'river', 'tunnel'}
FORWARD, BACKWARD = ({}, {})
for edge in connections:
    if edge['from_type'] in HYDRAULIC_KINDS and edge['to_type'] in HYDRAULIC_KINDS:
        left = (edge['from_type'], edge['from'])
        right = (edge['to_type'], edge['to'])
        FORWARD.setdefault(left, set()).add(right)
        BACKWARD.setdefault(right, set()).add(left)


In [ ]:
PLANT_FOR = {g: next((
    e['to']
    for e in connections
    if e['from'] == g and e['from_type'] == 'generator' and (e['to_type'] == 'plant')
)) for g in GENERATORS}
PLANT_GENERATORS = {p: [g for g in GENERATORS if PLANT_FOR[g] == p] for p in topology['model']['plant']}
PLANT_FEEDERS = {p: FEEDERS[gens[0]] for p, gens in PLANT_GENERATORS.items()}

def reachable(start, adjacency, stop_at_reservoir=False):
    """Walk named hydraulic connections, optionally stopping at the next reservoir."""
    pending, seen = (list(adjacency.get(start, [])), set())
    while pending:
        node = pending.pop()
        if node == start or node in seen:
            continue
        seen.add(node)
        if not (stop_at_reservoir and node[0] == 'reservoir'):
            pending.extend(adjacency.get(node, []))
    return seen
RECEIVERS = {p: sorted((name for kind, name in reachable(('plant', p), FORWARD, True) if kind == 'reservoir')) for p in PLANT_GENERATORS}
UPSTREAM_PLANTS = {p: sorted((name for kind, name in reachable(('plant', p), BACKWARD) if kind == 'plant')) for p in PLANT_GENERATORS}
DOWNSTREAM_PLANTS = {p: sorted((name for kind, name in reachable(('plant', p), FORWARD) if kind == 'plant')) for p in PLANT_GENERATORS}
LEVEL_CURVES = {r: (
    np.asarray(topology['model']['reservoir'][r]['vol_head']['x'], dtype=float),
    np.asarray(topology['model']['reservoir'][r]['vol_head']['y'], dtype=float),
) for r in RESERVOIRS}
GEN_SPECS = topology['model']['generator']


## Economic and topology features

The base inputs describe prices, calendar, water availability and environmental
requirements. Named upstream/downstream connections add reservoir filling,
water-value differences and local hydraulic context.


In [ ]:
def economic_context(margin, price):
    """Describe profitable hours and price patterns within each planning week."""
    margin, price = (np.asarray(margin, dtype=np.float64), np.asarray(price, dtype=np.float64))
    n, width = margin.shape
    positive = margin > 0
    positive_margin = np.maximum(margin, 0)
    left_length = np.zeros_like(margin)
    right_length = np.zeros_like(margin)
    left_sum = np.zeros_like(margin)
    right_sum = np.zeros_like(margin)
    next_positive = np.full_like(margin, HOURS)
    previous_positive = np.full_like(margin, HOURS)
    # Each recurrence resets independently for every scheduling case.
    for h in range(width):
        left_length[:, h] = np.where(positive[:, h], (left_length[:, h - 1] if h else 0) + 1, 0)
        left_sum[:, h] = np.where(positive[:, h], (left_sum[:, h - 1] if h else 0) + margin[:, h], 0)
        previous_positive[:, h] = np.where(positive[:, h], 0, np.minimum((previous_positive[:, h - 1] if h else HOURS) + 1, HOURS))
    for h in range(width - 1, -1, -1):
        right_length[:, h] = np.where(positive[:, h], (right_length[:, h + 1] if h < width - 1 else 0) + 1, 0)
        right_sum[:, h] = np.where(positive[:, h], (right_sum[:, h + 1] if h < width - 1 else 0) + margin[:, h], 0)
        next_positive[:, h] = np.where(
            positive[:, h],
            0,
            np.minimum((next_positive[:, h + 1] if h < width - 1 else HOURS) + 1, HOURS),
        )
    suffix_count = np.arange(width, 0, -1)
    suffix_positive = positive[:, ::-1].cumsum(1)[:, ::-1]
    suffix_margin = positive_margin[:, ::-1].cumsum(1)[:, ::-1]
    result = {
        'economic_margin_over_week_price_scale': margin / (price.std(1, keepdims=True) + 1.0),
        'economic_positive_streak_elapsed_h': left_length,
        'economic_positive_streak_remaining_h': right_length,
        'economic_positive_streak_total_h': left_length + right_length - positive,
        'economic_positive_streak_margin_sum': left_sum + right_sum - positive_margin,
        'economic_hours_to_next_positive': next_positive,
        'economic_hours_since_previous_positive': previous_positive,
        'economic_positive_fraction_remaining': suffix_positive / suffix_count,
        'economic_positive_margin_remaining_mean': suffix_margin / suffix_count,
        'economic_better_remaining_margin': np.maximum.accumulate(margin[:, ::-1], axis=1)[:, ::-1] - margin,
    }
    indices = np.arange(width)
    for window in (6, 24):
        end = np.minimum(indices + window, width)
        start = np.maximum(indices - window + 1, 0)
        prefix = np.concatenate([np.zeros((n, 1)), margin.cumsum(1)], axis=1)
        result[f'economic_next_{window}h_margin_mean'] = (prefix[:, end] - prefix[:, indices]) / (end - indices)
        result[f'economic_previous_{window}h_margin_mean'] = (prefix[:, indices + 1] - prefix[:, start]) / (indices + 1 - start)
    return result

def water_context(initial, cumulative, remaining, limits, capacities):
    """Describe stored water, incoming water and distance to reservoir limits."""
    initial = np.asarray(initial, dtype=np.float64)
    cumulative = np.asarray(cumulative, dtype=np.float64)
    remaining = np.asarray(remaining, dtype=np.float64)
    limits = np.asarray(limits, dtype=np.float64)
    capacities = np.asarray(capacities, dtype=np.float64)
    capacity = capacities.sum()
    start = initial[:, None, :]
    no_release = start + cumulative
    end_no_release = no_release + remaining
    headroom = np.maximum(capacities - start, 0)
    # A 1% capacity floor bounds the ratio near a full reservoir; apply per reservoir
    # before aggregation so spare space elsewhere does not cancel local pressure.
    pressure = np.log1p((cumulative + remaining) / np.maximum(headroom, 0.01 * capacities))
    future_floor = np.maximum.accumulate(limits[:, ::-1, :], axis=1)[:, ::-1, :]
    return {
        'water_week_inflow_to_headroom_log_max': pressure.max(2),
        'water_week_inflow_to_headroom_log_weighted': (pressure * capacities).sum(2) / capacity,
        'water_no_release_excess_fraction': np.maximum(no_release - capacities, 0).sum(2) / capacity,
        'water_end_no_release_excess_fraction': np.maximum(end_no_release - capacities, 0).sum(2) / capacity,
        'water_no_release_excess_max_fraction': np.maximum((no_release - capacities) / capacities, 0).max(2),
        'water_initial_margin_to_floor_fraction': (start - limits).sum(2) / capacity,
        'water_initial_margin_to_future_floor_fraction': (start - future_floor).sum(2) / capacity,
        'water_no_release_floor_deficit_fraction': np.maximum(limits - no_release, 0).sum(2) / capacity,
        'water_no_release_min_margin_fraction': ((no_release - limits) / capacities).min(2),
    }

def lookup(key, timestamps, columns=None):
    """Read known inputs at the requested UTC timestamps."""
    frame = tables[key] if columns is None else tables[key][columns]
    selected = frame.reindex(pd.DatetimeIndex(timestamps))
    if selected.isna().any().any():
        raise ValueError(f'Missing values while looking up {key}')
    return selected.to_numpy(dtype=np.float32)


In [ ]:
def base_features(cases):
    """Build hourly price, calendar, water and constraint features."""
    starts = pd.DatetimeIndex(cases['starttime'])
    n = len(starts)
    # Use datetime arithmetic rather than assuming pandas timestamps use nanoseconds.
    timestamps = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(np.tile(np.arange(HOURS), n), unit='h')
    price = lookup('prices', timestamps).reshape(n, HOURS)
    initial = lookup('volumes', starts, RESERVOIRS)
    terminal = lookup('water_values', starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflows = lookup('inflows', timestamps).reshape(n, HOURS, -1)
    features = {}

    def add(name, values):
        values = np.asarray(values, dtype=np.float32)
        if values.shape == (n,):
            values = np.repeat(values, HOURS)
        elif values.shape == (n, HOURS):
            values = values.reshape(-1)
        features[name] = values
    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add('horizon_hour', h)
    add('hours_remaining', HOURS - 1 - h)
    add('hour_of_day', timestamps.hour.to_numpy())
    add('day_of_week', timestamps.dayofweek.to_numpy())
    add('month', timestamps.month.to_numpy())
    add('day_of_year_sin', np.sin(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add('day_of_year_cos', np.cos(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add('price', price)
    for stat, values in (('mean', price.mean(1)), ('std', price.std(1)), ('min', price.min(1)), ('max', price.max(1))):
        add(f'price_week_{stat}', values)
    add('price_minus_week_mean', price - price.mean(1, keepdims=True))
    add('price_week_rank', pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, values in (('mean', daily.mean(2)), ('min', daily.min(2)), ('max', daily.max(2))):
        add(f'price_day_{stat}', np.repeat(values, 24, axis=1))
    add('price_prefix_mean', price.cumsum(1) / np.arange(1, HOURS + 1))
    add('price_suffix_mean', price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    # Clip at the case boundary, not at the global dataset boundary.
    for offset in (-24, -6, -1, 1, 6, 24):
        add(f'price_offset_{offset:+d}', price[:, np.clip(np.arange(HOURS) + offset, 0, HOURS - 1)])
    for j, reservoir in enumerate(RESERVOIRS):
        add(f'initial_volume__{reservoir}', initial[:, j])
        add(f'initial_fraction__{reservoir}', initial[:, j] / CAPACITY[reservoir])
        add(f'terminal_value__{reservoir}', terminal[:, j])
        add(f'price_minus_value__{reservoir}', price - terminal[:, j, None])
    for j, name in enumerate(tables['inflows'].columns):
        flow = inflows[:, :, j]
        cumulative = flow.cumsum(1) * (3600 / 1000000.0)
        add(f'inflow__{name}', flow)
        add(f'inflow_week_mean__{name}', flow.mean(1))
        add(f'inflow_cumulative_Mm3__{name}', cumulative)
        add(f'inflow_remaining_Mm3__{name}', cumulative[:, -1:] - cumulative)
    for key in ('min_flow', 'min_volume'):
        limits = lookup(key, timestamps).reshape(n, HOURS, -1)
        for j, name in enumerate(tables[key].columns):
            add(f'{key}__{name}', limits[:, :, j])
            add(f'{key}_week_max__{name}', limits[:, :, j].max(1))
            if key == 'min_volume':
                add(f'initial_margin_to_min__{name}', initial[:, RESERVOIRS.index(name), None] - limits[:, :, j])
    return pd.DataFrame(features, dtype=np.float32)


In [ ]:
def generator_features(base, generator):
    """Add the economic and water context of one generator's intake."""
    # Copy only one generator's matrix at a time; never materialize all 14 together.
    X = base.copy()
    feeders = FEEDERS[generator]
    capacity = sum((CAPACITY[r] for r in feeders))
    total_initial = sum((base[f'initial_volume__{r}'] for r in feeders))
    value = sum((base[f'terminal_value__{r}'] * CAPACITY[r] for r in feeders)) / capacity
    cumulative = sum((base[f'inflow_cumulative_Mm3__{r}'] for r in feeders))
    remaining = sum((base[f'inflow_remaining_Mm3__{r}'] for r in feeders))
    X['local_initial_fraction'] = total_initial / capacity
    X['local_terminal_value_capacity_weighted'] = value
    X['local_price_minus_terminal_value'] = base['price'] - value
    X['local_inflow'] = sum((base[f'inflow__{r}'] for r in feeders))
    X['local_cumulative_inflow_fraction'] = cumulative / capacity
    X['local_remaining_inflow_fraction'] = remaining / capacity
    X['local_available_water_proxy_fraction'] = (total_initial + cumulative + remaining) / capacity
    extra = {}
    margin = (base['price'] - value).to_numpy().reshape(-1, HOURS)
    extra.update(economic_context(margin, base['price'].to_numpy().reshape(-1, HOURS)))

    def cube(prefix):
        return np.stack([base[f'{prefix}__{r}'].to_numpy().reshape(-1, HOURS) for r in feeders], axis=2)
    initial = cube('initial_volume')[:, 0, :]
    cumulative_cube = cube('inflow_cumulative_Mm3')
    remaining_cube = cube('inflow_remaining_Mm3')
    floor = np.stack(
        [
            base[f'min_volume__{r}'].to_numpy().reshape(-1, HOURS) if f'min_volume__{r}' in base else np.zeros_like(cumulative_cube[:, :, 0])
            for r in feeders
        ],
        axis=2,
    )
    extra.update(water_context(initial, cumulative_cube, remaining_cube, floor, [CAPACITY[r] for r in feeders]))
    if extra:
        X = pd.concat(
            [
                X,
                pd.DataFrame(
                    {name: values.reshape(-1).astype(np.float32) for name, values in extra.items()},
                    index=X.index,
                ),
            ],
            axis=1,
        )
    return X.astype(np.float32)

def feature_matrix(cases):
    """Build hourly price, calendar, water and constraint features."""
    frame = base_features(cases)
    # Internal alignment metadata never enters the numeric feature matrix.
    frame.attrs['case_starts'] = pd.DatetimeIndex(cases['starttime'])
    return frame

def state(base, reservoirs, prefix):
    """Arrange reservoir values as case-by-hour-by-reservoir arrays."""
    return np.stack([base[f'{prefix}__{r}'].to_numpy().reshape(-1, HOURS) for r in reservoirs], axis=2)

def plant_state(base, plant):
    """Estimate initial intake level, tailwater level and available head."""
    feeders = PLANT_FEEDERS[plant]
    weights = np.array([CAPACITY[r] for r in feeders], dtype=np.float64)
    volumes = state(base, feeders, 'initial_volume')
    values = state(base, feeders, 'terminal_value')
    level = np.stack([np.interp(volumes[:, :, i], *LEVEL_CURVES[r]) for i, r in enumerate(feeders)], axis=2)
    intake_level = np.average(level, weights=weights, axis=2)
    receiver = RECEIVERS[plant]
    outlet = float(topology['model']['plant'][plant]['outlet_line'])
    tailwater = np.full_like(intake_level, outlet)
    if receiver:
        receiver_levels = np.stack(
            [
                np.interp(base[f'initial_volume__{r}'].to_numpy().reshape(-1, HOURS), *LEVEL_CURVES[r])
                for r in receiver
            ],
            axis=2,
        )
        tailwater = np.maximum(tailwater, receiver_levels.max(axis=2))
    return {
        'head': np.maximum(intake_level - tailwater, 1.0),
        'intake_level': intake_level,
        'tailwater': tailwater,
        'fraction': volumes.sum(axis=2) / weights.sum(),
        'value': np.average(values, weights=weights, axis=2),
    }


In [ ]:
def topology_features(base, generator):
    """Describe the reservoirs and plants connected to one generator."""
    plant = PLANT_FOR[generator]
    feeders, upstream, receiver = (FEEDERS[generator], UPSTREAM[generator], RECEIVERS[plant])
    local = plant_state(base, plant)
    price = base['price'].to_numpy().reshape(-1, HOURS)
    capacity = np.array([CAPACITY[r] for r in upstream])
    volumes = state(base, upstream, 'initial_volume')
    fractions = volumes / capacity
    terminal = state(base, upstream, 'terminal_value')
    cumulative = state(base, upstream, 'inflow_cumulative_Mm3')
    remaining = state(base, upstream, 'inflow_remaining_Mm3')
    root_capacity = capacity.sum()
    pressure = (cumulative + remaining) / np.maximum(capacity - volumes, 0.01 * capacity)
    extra = {
        'topo_initial_intake_level_m': local['intake_level'],
        'topo_initial_tailwater_m': local['tailwater'],
        'topo_initial_gross_head_m': local['head'],
        'topo_head_relative_reference': local['head'] / np.mean([c['ref'] for c in GEN_SPECS[generator]['turb_eff_curves']]),
        'topo_upstream_initial_fraction': volumes.sum(2) / root_capacity,
        'topo_upstream_min_fraction': fractions.min(2),
        'topo_upstream_max_fraction': fractions.max(2),
        'topo_upstream_std_fraction': fractions.std(2),
        'topo_upstream_value': np.average(terminal, weights=capacity, axis=2),
        'topo_upstream_value_min': terminal.min(2),
        'topo_upstream_value_max': terminal.max(2),
        'topo_upstream_value_std': terminal.std(2),
        'topo_upstream_inflow': state(base, upstream, 'inflow').sum(2),
        'topo_upstream_cumulative_fraction': cumulative.sum(2) / root_capacity,
        'topo_upstream_remaining_fraction': remaining.sum(2) / root_capacity,
        'topo_upstream_water_pressure_max': np.log1p(pressure).max(2),
        'topo_upstream_excess_fraction': np.maximum(volumes + cumulative + remaining - capacity, 0).sum(2) / root_capacity,
        'topo_upstream_price_minus_value': price - np.average(terminal, weights=capacity, axis=2),
    }
    for label, reservoirs in [('receiver', receiver), ('remote_upstream', sorted(set(upstream) - set(feeders)))]:
        if reservoirs:
            cap = np.array([CAPACITY[r] for r in reservoirs])
            vol = state(base, reservoirs, 'initial_volume')
            wv = np.average(state(base, reservoirs, 'terminal_value'), weights=cap, axis=2)
            frac = vol.sum(2) / cap.sum()
            inflow = state(base, reservoirs, 'inflow').sum(2)
            free = np.maximum(cap - vol, 0).sum(2) / cap.sum()
        else:
            frac = wv = inflow = free = np.zeros_like(price)
        extra.update({
            f'topo_{label}_fraction': frac,
            f'topo_{label}_value': wv,
            f'topo_{label}_inflow': inflow,
            f'topo_{label}_free_fraction': free,
            f'topo_local_minus_{label}_value': local['value'] - wv,
        })
    for direction, related in [('upstream', UPSTREAM_PLANTS[plant]), ('downstream', DOWNSTREAM_PLANTS[plant])]:
        related_states = [plant_state(base, p) for p in related]
        margins = np.stack([price - s['value'] for s in related_states], axis=2) if related_states else np.zeros((*price.shape, 1))
        fills = np.stack([s['fraction'] for s in related_states], axis=2) if related_states else np.zeros((*price.shape, 1))
        extra.update({
            f'topo_{direction}_plant_margin_mean': margins.mean(2),
            f'topo_{direction}_plant_margin_min': margins.min(2),
            f'topo_{direction}_plant_margin_max': margins.max(2),
            f'topo_{direction}_plant_positive_fraction': (margins > 0).mean(2),
            f'topo_{direction}_plant_fill_mean': fills.mean(2),
            f'topo_{direction}_plant_fill_min': fills.min(2),
            f'topo_{direction}_plant_fill_max': fills.max(2),
        })
    for r in RESERVOIRS:
        extra[f'topo_initial_level_m__{r}'] = np.interp(base[f'initial_volume__{r}'].to_numpy().reshape(-1, HOURS), *LEVEL_CURVES[r])
    # The source of each min-flow constraint is explicit in the hydraulic graph.
    for label, reservoirs in [('intake', set(feeders)), ('upstream', set(upstream))]:
        river_names = {name for r in reservoirs for kind, name in FORWARD.get(('reservoir', r), []) if kind == 'river'}
        requirement = sum(
            (
                base[f'min_flow__{r}'].to_numpy().reshape(-1, HOURS)
                for r in river_names
                if f'min_flow__{r}' in base
            ),
            np.zeros_like(price),
        )
        flow = state(base, sorted(reservoirs), 'inflow').sum(2)
        extra[f'topo_{label}_minflow_requirement'] = requirement
        extra[f'topo_{label}_natural_inflow_minus_requirement'] = flow - requirement
    return extra


In [ ]:
def efficiency(spec, head, power):
    """Estimate discharge in m3/s and efficiency at the supplied head and power."""
    head = np.asarray(head, dtype=np.float64)
    curves = sorted(spec['turb_eff_curves'], key=lambda curve: curve['ref'])
    reference_heads = np.array([curve['ref'] for curve in curves])
    discharge = power * 1000000.0 / (1000 * 9.81 * head * 0.8)
    generator_curve = spec['gen_eff_curve']
    generator_efficiency = np.interp(power, generator_curve['x'], np.array(generator_curve['y']) / 100)
    for _ in range(4):
        turbine_efficiencies = np.stack(
            [np.interp(discharge, curve['x'], np.array(curve['y']) / 100) for curve in curves],
            axis=-1,
        )
        upper = np.clip(np.searchsorted(reference_heads, head, side='right'), 1, max(1, len(reference_heads) - 1))
        if len(reference_heads) == 1:
            turbine_efficiency = turbine_efficiencies[..., 0]
        else:
            lower = upper - 1
            fraction = np.clip((head - reference_heads[lower]) / (reference_heads[upper] - reference_heads[lower]), 0, 1)
            turbine_efficiency = (
                np.take_along_axis(turbine_efficiencies, lower[..., None], axis=-1)[..., 0] * (1 - fraction)
                + np.take_along_axis(turbine_efficiencies, upper[..., None], axis=-1)[..., 0] * fraction
            )
        combined_efficiency = np.clip(generator_efficiency * turbine_efficiency, 0.3, 1.0)
        discharge = power * 1000000.0 / (1000 * 9.81 * head * combined_efficiency)
    return (discharge, combined_efficiency)


## LP allocation, projected head and water-value sensitivity

A continuous LP allocates water in six-hour blocks. It supplies own-unit and
plant allocation summaries, projected reservoir levels and turbine head.
Additional solves perturb terminal water values by +/-5%. These are approximate
physical descriptors, not future observed storage or commitment labels.
All selected features are included for Lio as well.


In [ ]:
LP_STEP = 6
LP_CACHE = {}
LP_STRUCTURE = None
PLANTS = list(PLANT_GENERATORS)

def conversion_plants(node, seen=None):
    """Follow the supplied generating route downstream from a node."""
    seen = set() if seen is None else seen
    if node in seen:
        raise ValueError('Cycle in water-value conversion route.')
    if node[0] == 'plant':
        return {node[1]}
    outgoing = [
        (e['to_type'], e['to'])
        for e in connections
        if (e['from_type'], e['from']) == node and e['to_type'] in HYDRAULIC_KINDS
    ]
    if node[0] == 'reservoir':
        direct = [t for t in outgoing if t[0] == 'plant']
        tunnels = [t for t in outgoing if t[0] == 'tunnel']
        rivers = [t for t in outgoing if t[0] == 'river']
        if direct or tunnels:
            outgoing = direct or tunnels
        elif rivers:
            main = [t for t in rivers if int(topology['model']['river'][t[1]].get('main_river', 0)) == 1]
            outgoing = main or rivers[:1]
    return set().union(*(conversion_plants(t, seen | {node}) for t in outgoing)) if outgoing else set()

def make_lp_structure():
    """Build water-balance and storage-limit matrices for the six-hour LP."""
    from scipy.sparse import lil_matrix
    nodes = sorted(set(FORWARD) | {node for targets in FORWARD.values() for node in targets})
    raw_arcs = sorted((
        (source_node, destination_node)
        for source_node, targets in FORWARD.items()
        for destination_node in targets
    ))
    raw_arcs += [(node, ('sink', 'outside')) for node in nodes if not FORWARD.get(node)]
    arcs = []
    for left, right in raw_arcs:
        if left[0] == 'plant':
            outgoing = [destination_node for source_node, destination_node in raw_arcs if source_node == left]
            arcs.extend(((left, right, generator) for generator in PLANT_GENERATORS[left[1]]))
        else:
            arcs.append((left, right, None))
    block_count, reservoir_count, arc_count, node_count = (HOURS // LP_STEP, len(RESERVOIRS), len(arcs), len(nodes))
    positions = {node: i for i, node in enumerate(nodes)}
    variable_count = block_count * (arc_count + 3 * reservoir_count)
    balance_matrix = lil_matrix((block_count * node_count, variable_count), dtype=float)
    limit_matrix = lil_matrix((2 * block_count * reservoir_count, variable_count), dtype=float)
    for block in range(block_count):
        for index, (source_node, destination_node, generator) in enumerate(arcs):
            balance_matrix[block * node_count + positions[source_node], block * arc_count + index] += 1
            if destination_node in positions:
                balance_matrix[block * node_count + positions[destination_node], block * arc_count + index] -= 1
        for index, reservoir in enumerate(RESERVOIRS):
            column = block_count * arc_count + block * reservoir_count + index
            row = block * node_count + positions['reservoir', reservoir]
            balance_matrix[row, column] += 1
            if block:
                balance_matrix[row, column - reservoir_count] -= 1
            upper, lower = (2 * (block * reservoir_count + index), 2 * (block * reservoir_count + index) + 1)
            limit_matrix[upper, column], limit_matrix[lower, column] = (1, -1)
            limit_matrix[upper, block_count * (arc_count + reservoir_count) + block * reservoir_count + index] = -1
            limit_matrix[lower, block_count * (arc_count + 2 * reservoir_count) + block * reservoir_count + index] = -1
    unit_arc = {generator: index for index, (_, _, generator) in enumerate(arcs) if generator is not None}
    return dict(
        steps=block_count,
        nr=reservoir_count,
        nf=arc_count,
        nn=node_count,
        nv=variable_count,
        nodes=nodes,
        positions=positions,
        arcs=arcs,
        eq=balance_matrix.tocsr(),
        ub=limit_matrix.tocsr(),
        unit_arc=unit_arc,
    )

def heads_from_volume(volume):
    """Convert initial or projected storage into plant head in metres."""
    levels = {r: np.interp(volume[..., j], *LEVEL_CURVES[r]) for j, r in enumerate(RESERVOIRS)}
    heads = {}
    for p in PLANTS:
        intake = np.average(
            np.stack([levels[r] for r in PLANT_FEEDERS[p]], axis=-1),
            weights=[CAPACITY[r] for r in PLANT_FEEDERS[p]],
            axis=-1,
        )
        tail = np.full_like(intake, float(topology['model']['plant'][p]['outlet_line']))
        for r in RECEIVERS[p]:
            tail = np.maximum(tail, levels[r])
        heads[p] = np.maximum(intake - tail, 1.0)
    return heads


In [ ]:
def terminal_conversion(volume, water_values):
    """Approximate the downstream monetary value of water left at week end."""
    initial_heads = heads_from_volume(volume)
    factors = {}
    for p in PLANTS:
        grid = [
            np.linspace(float(GEN_SPECS[g]['p_min']), float(GEN_SPECS[g]['p_max']), 33)
            for g in PLANT_GENERATORS[p]
        ]
        q = [
            efficiency(GEN_SPECS[g], np.full(33, initial_heads[p]), power)[0]
            for g, power in zip(PLANT_GENERATORS[p], grid)
        ]
        factors[p] = float(np.max(sum(grid) * 1000000.0 / (sum(q) * 3600)))
    selected = {}
    for r in RESERVOIRS:
        candidates = conversion_plants(('reservoir', r))
        selected[r] = max(candidates, key=lambda p: (factors[p], p))
    local = dict(zip(RESERVOIRS, water_values))
    money = {}

    def recurse(r, seen=None):
        if r in money:
            return money[r]
        seen = set() if seen is None else seen
        if r in seen:
            raise ValueError('Cycle in downstream terminal valuation.')
        p = selected[r]
        receiving = max([0.0] + [recurse(t, seen | {r}) for t in RECEIVERS[p]])
        money[r] = float(local[r]) * factors[p] + receiving
        return money[r]
    return (np.array([recurse(r) for r in RESERVOIRS]), factors)


In [ ]:
def solve_unit_lp(start, projected=None, value_multiplier=1.0):
    """Allocate water across the week using a continuous linear program."""
    from scipy.optimize import linprog
    import warnings
    global LP_STRUCTURE
    if LP_STRUCTURE is None:
        LP_STRUCTURE = make_lp_structure()
    structure = LP_STRUCTURE
    block_count, reservoir_count, arc_count, node_count = [structure[x] for x in ('steps', 'nr', 'nf', 'nn')]
    times = pd.date_range(start, periods=HOURS, freq='h')
    initial = lookup('volumes', [start], RESERVOIRS)[0].astype(float)
    values = lookup('water_values', [start + pd.Timedelta(hours=HOURS)], RESERVOIRS)[0].astype(float)
    price = lookup('prices', times).reshape(block_count, LP_STEP).mean(1)
    inflows = tables['inflows'].reindex(times)
    minflows = tables['min_flow'].reindex(times)
    minvolumes = tables['min_volume'].reindex(times)
    block_volumes = np.tile(initial, (block_count, 1)) if projected is None else np.vstack([initial, projected[:-1]])
    heads = heads_from_volume(block_volumes)
    qmax, factor = ({}, {})
    for generator in GENERATORS:
        power = float(GEN_SPECS[generator]['p_max'])
        qmax[generator] = efficiency(GEN_SPECS[generator], heads[PLANT_FOR[generator]], power)[0]
        factor[generator] = power * 1000000.0 / (qmax[generator] * 3600)
    money, best_factor = terminal_conversion(initial, values * value_multiplier)
    objective = np.zeros(structure['nv'])
    water_supply = np.zeros(block_count * node_count)
    limits = np.zeros(2 * block_count * reservoir_count)
    bounds = []
    seconds = LP_STEP * 3600
    for step in range(block_count):
        for index, (left, right, generator) in enumerate(structure['arcs']):
            low, high = (0.0, 1000.0 * seconds / 1000000.0)
            if generator is not None:
                high = qmax[generator][step] * seconds / 1000000.0
                objective[step * arc_count + index] = -price[step] * factor[generator][step]
            if left[0] == 'river' and left[1] in minflows:
                low = float(minflows[left[1]].to_numpy().reshape(block_count, LP_STEP)[step].sum()) * 3600 / 1000000.0
            bounds.append((low, max(high, low)))
        for node, position in structure['positions'].items():
            if node[0] in {'reservoir', 'river'} and node[1] in inflows:
                water_supply[step * node_count + position] = float(inflows[node[1]].to_numpy().reshape(block_count, LP_STEP)[step].sum()) * 3600 / 1000000.0
        for index, reservoir in enumerate(RESERVOIRS):
            if step == 0:
                water_supply[structure['positions']['reservoir', reservoir]] += initial[index]
            limits[2 * (step * reservoir_count + index)] = CAPACITY[reservoir]
            floor = float(minvolumes[reservoir].to_numpy().reshape(block_count, LP_STEP)[step].max()) if reservoir in minvolumes else 0.0
            limits[2 * (step * reservoir_count + index) + 1] = -floor
    bounds.extend([(0, None)] * (3 * block_count * reservoir_count))
    objective[block_count * arc_count + (block_count - 1) * reservoir_count:block_count * (arc_count + reservoir_count)] = -money
    objective[block_count * (arc_count + reservoir_count):] = 1000000.0
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='Unrecognized options detected')
        solution = linprog(
            objective,
            A_eq=structure['eq'],
            b_eq=water_supply,
            A_ub=structure['ub'],
            b_ub=limits,
            bounds=bounds,
            method='highs',
            options={'threads': 1, 'time_limit': 20.0},
        )
    if not solution.success:
        raise RuntimeError(f'Unit LP failed at {start}: {solution.message}')
    residual = float(np.max(np.abs(structure['eq'] @ solution.x - water_supply)))
    if residual > 1e-05:
        raise ValueError('LP water-balance residual exceeds tolerance.')
    flows = solution.x[:block_count * arc_count].reshape(block_count, arc_count)
    fractions = np.column_stack([
        flows[:, structure['unit_arc'][generator]] / (qmax[generator] * seconds / 1000000.0)
        for generator in GENERATORS
    ])
    storage = solution.x[block_count * arc_count:block_count * (arc_count + reservoir_count)].reshape(
        block_count,
        reservoir_count,
    )
    slack = solution.x[block_count * (arc_count + reservoir_count):].reshape(2, block_count, reservoir_count).sum(axis=(0, 2))
    shadows = solution.eqlin.marginals.reshape(block_count, node_count)
    margins = np.column_stack([
        (price * factor[generator] + shadows[:, structure['positions']['plant', PLANT_FOR[generator]]]) / factor[generator]
        for generator in GENERATORS
    ])
    water_cost = np.column_stack([
        (
            (
                shadows[:, structure['positions'][structure['arcs'][structure['unit_arc'][generator]][1]]]
                if structure['arcs'][structure['unit_arc'][generator]][1] in structure['positions']
                else 0.0
            )
            - shadows[:, structure['positions']['plant', PLANT_FOR[generator]]]
        )
        for generator in GENERATORS
    ])
    unit_energy = np.column_stack([factor[generator] for generator in GENERATORS])
    net_profit = price[:, None] * unit_energy - water_cost
    bounds_shadow = (
        solution.lower.marginals[:block_count * arc_count].reshape(block_count, arc_count)
        + solution.upper.marginals[:block_count * arc_count].reshape(block_count, arc_count)
    )
    kkt_error = float(np.max(np.abs(net_profit + np.column_stack([bounds_shadow[:, structure['unit_arc'][generator]] for generator in GENERATORS]))))
    return dict(
        fractions=fractions,
        storage=storage,
        margins=margins,
        slack=slack,
        heads=np.column_stack([heads[p] for p in PLANTS]),
        residual=residual,
        objective=float(solution.fun),
        terminal_money=money,
        unit_water_cost_EUR_per_Mm3=water_cost,
        unit_net_margin_EUR_per_MWh=net_profit / unit_energy,
        unit_qmax_m3s=np.column_stack([qmax[generator] for generator in GENERATORS]),
        unit_arc_kkt_error=kkt_error,
    )


In [ ]:
def lp_case(start):
    """Solve the initial-head, updated-head and two water-value scenarios for one case."""
    start = pd.Timestamp(start)
    if start not in LP_CACHE:
        base = solve_unit_lp(start)
        corrected = solve_unit_lp(start, projected=base['storage'])
        low = solve_unit_lp(start, projected=base['storage'], value_multiplier=0.95)
        high = solve_unit_lp(start, projected=base['storage'], value_multiplier=1.05)
        LP_CACHE[start] = dict(base=base, corrected=corrected, low=low, high=high)
    return LP_CACHE[start]

def precompute_lp(starts):
    """Prepare independent case LPs using four workers and one solver thread each."""
    from concurrent.futures import ThreadPoolExecutor
    global LP_STRUCTURE
    if LP_STRUCTURE is None:
        LP_STRUCTURE = make_lp_structure()
    missing = [s for s in pd.DatetimeIndex(starts).unique() if s not in LP_CACHE]
    with ThreadPoolExecutor(max_workers=4) as pool:
        for i, _ in enumerate(pool.map(lp_case, missing)):
            if (i + 1) % 100 == 0:
                print(f'LP cases {i + 1}/{len(missing)}', flush=True)

def relative_features(base):
    """Scale price and water-value margins within each case."""
    price = base['price'].to_numpy().reshape(-1, HOURS)
    local = {p: plant_state(base, p)['value'] for p in PLANTS}
    values = np.stack([local[p][:, 0] for p in PLANTS], axis=1)
    scale = np.maximum(np.median(np.abs(np.concatenate([price, values], axis=1)), axis=1), 1.0)[:, None]
    result = {}
    # Nine local opportunity values plus the eight available intake/receiver differences.
    # Explicit route names, rather than arbitrary row positions, define the pairs.
    for p in PLANTS:
        margin = (price - local[p]) / scale
        result[f'relative_local__{p}'] = margin
        result[f'relative_local_week_mean__{p}'] = np.repeat(margin.mean(1)[:, None], HOURS, axis=1)
    pairs = [p for p in PLANTS if RECEIVERS[p]]
    for p in pairs:
        receiver = RECEIVERS[p]
        receiving = np.average(state(base, receiver, 'terminal_value'), weights=[CAPACITY[r] for r in receiver], axis=2)
        # Water values are local EUR/MWh, so this is a relative-value descriptor,
        # not the exact profit or water cost of a cascade transfer.
        margin = (price - (local[p] - receiving)) / scale
        result[f'relative_net__{p}'] = margin
        result[f'relative_net_week_mean__{p}'] = np.repeat(margin.mean(1)[:, None], HOURS, axis=1)
    return result

def nine_summaries(values, prefix):
    """Summarize allocation level, duration and persistence within a week."""
    positive = values > 1e-07
    elapsed = np.zeros_like(values)
    remaining = np.zeros_like(values)
    for h in range(HOURS):
        elapsed[:, h] = np.where(positive[:, h], (elapsed[:, h - 1] if h else 0) + 1, 0)
    for h in range(HOURS - 1, -1, -1):
        remaining[:, h] = np.where(positive[:, h], (remaining[:, h + 1] if h < HOURS - 1 else 0) + 1, 0)
    return {f'{prefix}_{key}': value for key, value in {
        'fraction': values,
        'on_proxy': positive.astype(float),
        'week_mean': np.repeat(values.mean(1)[:, None], HOURS, axis=1),
        'remaining_mean': values[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1),
        'elapsed_mean': values.cumsum(1) / np.arange(1, HOURS + 1),
        'remaining_on_hours': positive[:, ::-1].cumsum(1)[:, ::-1],
        'elapsed_on_hours': positive.cumsum(1),
        'positive_streak_elapsed': elapsed,
        'positive_streak_remaining': remaining,
    }.items()}


In [ ]:
def hydraulic_features(base, generator, corrected=False):
    """Turn LP allocations, projected head and sensitivity scenarios into hourly inputs."""
    data = [lp_case(s) for s in base.attrs['case_starts']]
    j = GENERATORS.index(generator)
    p = PLANT_FOR[generator]
    repeat = lambda x: np.repeat(x, LP_STEP, axis=1)
    fractions = np.stack([x['base']['fractions'] for x in data])
    margins = np.stack([x['base']['margins'] for x in data])
    result = {}
    for plant in PLANTS:
        indices = [GENERATORS.index(g) for g in PLANT_GENERATORS[plant]]
        powers = np.array([float(GEN_SPECS[g]['p_max']) for g in PLANT_GENERATORS[plant]])
        result[f'lp_plant_fraction__{plant}'] = repeat(np.average(fractions[:, :, indices], weights=powers, axis=2))
        result[f'lp_plant_margin__{plant}'] = repeat(np.average(margins[:, :, indices], weights=powers, axis=2))
    result.update(nine_summaries(repeat(fractions[:, :, j]), 'lp_own'))
    if corrected:
        scenarios = {key: repeat(np.stack([x[key]['fractions'][:, j] for x in data])) for key in ('corrected', 'low', 'high')}
        result.update(nine_summaries(scenarios['corrected'], 'lp_head_own'))
        for key in ('low', 'high'):
            result[f'lp_{key}_value_fraction'] = scenarios[key]
            result[f'lp_{key}_value_on_proxy'] = (scenarios[key] > 1e-07).astype(float)
            result[f'lp_{key}_minus_head_fraction'] = scenarios[key] - scenarios['corrected']
        result['lp_head_minus_initial_fraction'] = scenarios['corrected'] - repeat(fractions[:, :, j])
        result['lp_head_gross_head_m'] = repeat(np.stack([x['corrected']['heads'][:, PLANTS.index(p)] for x in data]))
        storage = np.stack([x['corrected']['storage'] for x in data])
        for label, reservoirs in [('intake', FEEDERS[generator]), ('receiver', RECEIVERS[p])]:
            indices = [RESERVOIRS.index(r) for r in reservoirs]
            value = storage[:, :, indices].sum(2) / sum((CAPACITY[r] for r in reservoirs)) if indices else np.zeros(storage.shape[:2])
            result[f'lp_head_projected_{label}_fraction'] = repeat(value)
        result['lp_head_soft_violation_Mm3'] = repeat(np.stack([x['corrected']['slack'] for x in data]))
        result['lp_sensitivity_on_fraction'] = np.mean(np.stack([v > 1e-07 for v in scenarios.values()]), axis=0)
        if len(PLANT_GENERATORS[p]) > 1:
            indices = [GENERATORS.index(g) for g in PLANT_GENERATORS[p]]
            powers = [float(GEN_SPECS[g]['p_max']) for g in PLANT_GENERATORS[p]]
            plant_schedule = repeat(np.stack([np.average(x['corrected']['fractions'][:, indices], weights=powers, axis=1) for x in data]))
            result.update(nine_summaries(plant_schedule, 'lp_own_plant_on'))
    return result

def model_features(base, generator):
    """Combine the selected input groups for one generator."""
    frame = generator_features(base, generator)
    extra = topology_features(base, generator)
    extra.update(relative_features(base))
    extra.update(hydraulic_features(base, generator, corrected=True))
    result = pd.concat(
        [
            frame,
            pd.DataFrame(
                {k: np.asarray(v, dtype=np.float32).reshape(-1) for k, v in extra.items()},
                index=base.index,
            ),
        ],
        axis=1,
    )
    return result.astype(np.float32)


## Train the 14 classifiers

All labeled cases are used. The final tree counts are fixed, so no test labels
or early stopping are needed. Only one generator's feature matrices are
materialized at a time to limit memory. LP preprocessing and fitting use at
most four CPU threads.


In [ ]:
import lightgbm as lgb

def train_generators():
    """Train the 14 classifiers with fixed tree counts, then predict the test cases."""
    precompute_lp(pd.concat([train_cases.starttime, test_cases.starttime]))
    fit_base = feature_matrix(train_cases[['Run No', 'starttime']])
    test_base = feature_matrix(test_cases[['Run No', 'starttime']])
    predictions = {}
    # Build and fit one generator at a time to keep memory use bounded.
    for generator in GENERATORS:
        started = perf_counter()
        train_features = model_features(fit_base, generator)
        test_features = model_features(test_base, generator)
        if list(train_features.columns) != list(test_features.columns):
            raise ValueError('Training and prediction features differ.')
        dataset = lgb.Dataset(
            train_features.to_numpy(copy=False),
            label=labels_for(train_cases, generator),
            feature_name=[f'f{i:04d}' for i in range(len(train_features.columns))],
            free_raw_data=True,
        )
        # Fix every sampling seed for reproducible boosting.
        parameters = {
            **PARAMETERS,
            'seed': SEED,
            'bagging_seed': SEED,
            'feature_fraction_seed': SEED,
            'data_random_seed': SEED,
        }
        model = lgb.train(parameters, dataset, num_boost_round=TREE_COUNTS[generator])
        tree_count = int(model.best_iteration or model.current_iteration())
        probability = model.predict(test_features.to_numpy(copy=False), num_iteration=tree_count, num_threads=THREADS).astype(np.float32)
        predictions[generator] = probability.reshape(len(test_cases), HOURS)
        print(
            f'{generator}: {len(train_features.columns)} inputs, {tree_count} trees, {perf_counter() - started:.1f}s',
            flush=True,
        )
        # Release large feature matrices before the next generator.
        del model, dataset, train_features, test_features
        gc.collect()
    return predictions
predictions = train_generators()


## Save the prediction CSV

The label file defines the target columns. `prediction_mapping.csv` defines
the test IDs and row order. The final checks cover the table shape, IDs and
probability range.


In [ ]:
def assemble(cases, predictions):
    """Place each generator's hourly probabilities in the required column order."""
    output = np.empty((len(cases), len(LABELS)), dtype=np.float32)
    for g in GENERATORS:
        value = np.asarray(predictions[g])
        output[:, TARGET_POSITIONS[g]] = value
    frame = pd.DataFrame(output, columns=LABELS)
    frame.insert(0, 'Run No', cases['Run No'].to_numpy())
    return frame

def export_predictions(frame, path, reference_sha256):
    """Check the final table, preserve any earlier file, and write the CSV."""
    expected_columns = ['Run No', *LABELS]
    values = frame[LABELS].to_numpy()
    if list(frame.columns) != expected_columns or len(frame) != len(test_cases):
        raise ValueError('The prediction table has the wrong shape or columns.')
    if not np.array_equal(frame['Run No'], test_cases['Run No']):
        raise ValueError('Prediction IDs differ from the test mapping.')
    if not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
        raise ValueError('Predictions must be finite probabilities between 0 and 1.')
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        archive = path.parent / 'archive' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
        archive.mkdir(parents=True)
        shutil.copy2(path, archive / path.name)
    frame.to_csv(path, index=False)
    # The fingerprint verifies the complete file without reading an older CSV.
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print('CSV:', path)
    print('Shape:', frame.shape)
    print('Exact reference match:', digest == reference_sha256)
    return path
submission_path = export_predictions(assemble(test_cases, predictions), OUTPUT_PATH, EXPECTED_CSV_SHA256)
